# 538D widget render stress test

**Goal:** Find what makes `IntSlider`/`FloatSlider` views disappear while Dropdowns still show (Cursor + JupyterLab).

**How to run:** Restart kernel → Run All. Record per test: sliders visible? (Y/N)

| Test | What it isolates |
|------|------------------|
| 1 | Baseline: 1 slider + 1 dropdown |
| 2 | 12 sliders in one `VBox` (notebook-born) |
| 3 | Same 12 sliders via `exec()` string |
| 4 | Full playground mix (sliders + dropdowns + HTML), notebook-born |
| 5 | Same mix via `exec()` string |
| 6 | Mix + plot HTML img (valid PNG in `<img>` tag) |
| 7 | Two-column `HBox` like current playground layout |
| 8 | Probe mix in **nested CELL 10 layout** (no matplotlib) |
| 9 | Real `tier1_cell10_playground_run.py` (deferred redraw) |
| 10 | Test 4 mix + **immediate** `plt.close("all")` after display |
| **BISECT** | Incremental `tier1_cell10_playground_bisect.py` — re-run last cell only |


In [11]:
# Test 1 — baseline
import ipywidgets as widgets
from IPython.display import display

print("=== Test 1: 1 slider + 1 dropdown (separate display) ===")
display(widgets.IntSlider(value=5, min=0, max=10, description="slider"))
display(widgets.Dropdown(options=["a", "b"], value="a", description="dropdown"))


=== Test 1: 1 slider + 1 dropdown (separate display) ===


IntSlider(value=5, description='slider', max=10)

Dropdown(description='dropdown', options=('a', 'b'), value='a')

In [12]:
# Test 2 — 12 sliders, one VBox (notebook-born, not exec)
import ipywidgets as widgets
from IPython.display import display

print("=== Test 2: 12 sliders in one VBox (notebook-born) ===")
_sl = [
    widgets.IntSlider(value=100, min=4, max=2500, description="Teams J"),
    widgets.IntSlider(value=15, min=5, max=40, description="Roster size"),
    widgets.FloatSlider(value=0.05, min=0.05, max=2.0, description="tau"),
    widgets.FloatSlider(value=-1.85, min=-3, max=3, description="T low"),
    widgets.FloatSlider(value=7.4, min=-3, max=10, description="T high"),
    widgets.FloatSlider(value=0.0, min=0, max=2, description="pref alpha"),
    widgets.IntSlider(value=42, min=0, max=99999, description="Seed"),
    widgets.FloatSlider(value=0.71, min=-1, max=3.5, description="theta"),
    widgets.FloatSlider(value=18.0, min=1, max=40, description="gamma"),
    widgets.IntSlider(value=20, min=5, max=30, description="LOO bins"),
    widgets.IntSlider(value=200, min=5, max=2000, description="Selections K"),
    widgets.FloatSlider(value=0.1, min=0, max=1, description="weight w"),
]
display(widgets.VBox(_sl))
print(f"Displayed {len(_sl)} sliders in one VBox")


=== Test 2: 12 sliders in one VBox (notebook-born) ===


Displayed 12 sliders in one VBox


In [13]:
# Test 3 — 12 sliders via exec() (same widgets, exec-born)
print("=== Test 3: 12 sliders in one VBox (exec-born) ===")
_exec_src = '''
import ipywidgets as widgets
from IPython.display import display
_sl = [
    widgets.IntSlider(value=100, min=4, max=2500, description="Teams J"),
    widgets.IntSlider(value=15, min=5, max=40, description="Roster size"),
    widgets.FloatSlider(value=0.05, min=0.05, max=2.0, description="tau"),
    widgets.FloatSlider(value=-1.85, min=-3, max=3, description="T low"),
    widgets.FloatSlider(value=7.4, min=-3, max=10, description="T high"),
    widgets.FloatSlider(value=0.0, min=0, max=2, description="pref alpha"),
    widgets.IntSlider(value=42, min=0, max=99999, description="Seed"),
    widgets.FloatSlider(value=0.71, min=-1, max=3.5, description="theta"),
    widgets.FloatSlider(value=18.0, min=1, max=40, description="gamma"),
    widgets.IntSlider(value=20, min=5, max=30, description="LOO bins"),
    widgets.IntSlider(value=200, min=5, max=2000, description="Selections K"),
    widgets.FloatSlider(value=0.1, min=0, max=1, description="weight w"),
]
display(widgets.VBox(_sl))
print(f"exec: displayed {len(_sl)} sliders")
'''
exec(compile(_exec_src, "<test3>", "exec"), globals())


=== Test 3: 12 sliders in one VBox (exec-born) ===


exec: displayed 12 sliders


In [14]:
# Test 4 — playground-style mix (12 sliders + 7 dropdowns + HTML), notebook-born
import ipywidgets as widgets
from IPython.display import display

print("=== Test 4: full mix, notebook-born ===")
style = {"description_width": "168px"}
lay = widgets.Layout(width="460px")

def _slider(desc, val, wtype="int", **kw):
    cls = widgets.IntSlider if wtype == "int" else widgets.FloatSlider
    return cls(value=val, description=desc, style=style, layout=lay, continuous_update=False, **kw)

def _dd(desc, opts):
    return widgets.Dropdown(options=opts, value=opts[0], description=desc, style=style, layout=lay)

kids = [
    widgets.HTML("<b>Pools</b>"),
    _slider("Teams J", 100, wtype="int", min=4, max=2500),
    _slider("Roster", 15, wtype="int", min=5, max=40),
    _slider("tau", 0.05, wtype="float", min=0.05, max=2.0),
    _dd("Kernel", ["gaussian", "cauchy"]),
    _dd("Target T", ["uniform", "empirical_530"]),
    _slider("T low", -1.85, wtype="float"),
    _slider("T high", 7.4, wtype="float"),
    _slider("pref", 0.0, wtype="float"),
    _dd("A draw", ["normal", "empirical_530"]),
    _slider("Seed", 42, wtype="int", min=0, max=99999),
    widgets.HTML("<b>Selection</b>"),
    _dd("Pool L", ["quality", "crowding"]),
    _slider("theta", 0.71, wtype="float"),
    _slider("gamma", 18.0, wtype="float"),
    _slider("bins", 20, wtype="int", min=5, max=30),
    _dd("bin mode", ["quantile", "equal_width"]),
    _slider("K", 200, wtype="int", min=5, max=2000),
    _dd("score", ["loo_gap_plus_ability"]),
    _slider("w", 0.1, wtype="float", min=0, max=1),
    _dd("winner", ["A", "C"]),
]
display(widgets.VBox(kids, layout=widgets.Layout(align_items="flex-start")))
print(f"Mix: {len(kids)} children (sliders + dropdowns + HTML)")


=== Test 4: full mix, notebook-born ===


Mix: 21 children (sliders + dropdowns + HTML)


In [15]:
# Test 5 — same mix via exec()
print("=== Test 5: full mix, exec-born ===")
from pathlib import Path
_probe = Path("tier1_cell10_widget_probe.py")
if not _probe.is_file():
    _probe = Path("sports/tier1_cell10_widget_probe.py")
exec(compile(_probe.read_text(encoding="utf-8"), str(_probe), "exec"), globals())


=== Test 5: full mix, exec-born ===


probe exec: 21 children


In [16]:
# Test 6 — 12 sliders + plot HTML in one VBox (valid PNG base64)
import ipywidgets as widgets
from IPython.display import display

_MIN_PNG_B64 = (
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8z8BQDwAEhQGAhKmMIQAAAABJRU5ErkJggg=="
)

print("=== Test 6: 12 sliders + plot HTML in one VBox ===")
_sl6 = [widgets.IntSlider(value=i, min=0, max=10, description=f"s{i}") for i in range(12)]
_plot6 = widgets.HTML(
    value=(
        f'<img alt="plot" src="data:image/png;base64,{_MIN_PNG_B64}" '
        'style="display:block;max-width:400px;"/>'
    )
)
display(widgets.VBox(_sl6 + [_plot6]))
print("Test 6: expect a tiny red square below the sliders (not a broken icon)")


=== Test 6: 12 sliders + plot HTML in one VBox ===


Test 6: expect a tiny red square below the sliders (not a broken icon)


In [17]:
# Test 7 — two-column HBox with 12 sliders (notebook-born)
import ipywidgets as widgets
from IPython.display import display

print("=== Test 7: two-column HBox, 6+6 sliders ===")
_left = [widgets.FloatSlider(value=i / 10, description=f"L{i}") for i in range(6)]
_right = [widgets.FloatSlider(value=i / 10, description=f"R{i}") for i in range(6)]
display(
    widgets.HBox(
        [
            widgets.VBox(_left, layout=widgets.Layout(min_width="460px")),
            widgets.VBox(_right, layout=widgets.Layout(min_width="460px")),
        ],
        layout=widgets.Layout(flex_flow="row wrap"),
    )
)


=== Test 7: two-column HBox, 6+6 sliders ===


In [18]:
# Test 8 — probe widgets in real nested CELL 10 layout (no matplotlib)
print("=== Test 8: nested layout probe (exec) ===")
from pathlib import Path
_lay = Path("tier1_cell10_widget_probe_layout.py")
if not _lay.is_file():
    _lay = Path("sports/tier1_cell10_widget_probe_layout.py")
exec(compile(_lay.read_text(encoding="utf-8"), str(_lay), "exec"), globals())


=== Test 8: nested layout probe (exec) ===


layout probe: nested HBox + footer (no matplotlib)


In [19]:
# Test 10 — Test 4 mix + immediate matplotlib (mimics old playground redraw)
import base64
import io

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display

print("=== Test 10: mix + immediate plt.close('all') after display ===")
style = {"description_width": "168px"}
lay = widgets.Layout(width="460px")

def _slider(desc, val, wtype="int", **kw):
    cls = widgets.IntSlider if wtype == "int" else widgets.FloatSlider
    return cls(value=val, description=desc, style=style, layout=lay, continuous_update=False, **kw)

def _dd(desc, opts):
    return widgets.Dropdown(options=opts, value=opts[0], description=desc, style=style, layout=lay)

plot_holder = widgets.HTML(value="<i>(no plot yet)</i>")
kids = [
    widgets.HTML("<b>Pools</b>"),
    _slider("Teams J", 100, wtype="int", min=4, max=2500),
    _slider("Roster", 15, wtype="int", min=5, max=40),
    _slider("tau", 0.05, wtype="float", min=0.05, max=2.0),
    _dd("Kernel", ["gaussian", "cauchy"]),
    _dd("Target T", ["uniform", "empirical_530"]),
    _slider("T low", -1.85, wtype="float"),
    _slider("T high", 7.4, wtype="float"),
    _slider("pref", 0.0, wtype="float"),
    _dd("A draw", ["normal", "empirical_530"]),
    _slider("Seed", 42, wtype="int", min=0, max=99999),
    widgets.HTML("<b>Selection</b>"),
    _dd("Pool L", ["quality", "crowding"]),
    _slider("theta", 0.71, wtype="float"),
    _slider("gamma", 18.0, wtype="float"),
    _slider("bins", 20, wtype="int", min=5, max=30),
    _dd("bin mode", ["quantile", "equal_width"]),
    _slider("K", 200, wtype="int", min=5, max=2000),
    _dd("score", ["loo_gap_plus_ability"]),
    _slider("w", 0.1, wtype="float", min=0, max=1),
    _dd("winner", ["A", "C"]),
    plot_holder,
]
panel10 = widgets.VBox(kids, layout=widgets.Layout(align_items="flex-start"))
display(panel10)
was_interactive = plt.isinteractive()
plt.ioff()
try:
    plt.close("all")
    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    ax.plot([0, 1, 2], [0, 1, 0], color="steelblue")
    ax.set_title("Test 10 matplotlib burst")
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    plot_holder.value = f'<img alt="plot" src="data:image/png;base64,{b64}" style="max-width:400px;"/>'
finally:
    if was_interactive:
        plt.ion()
    else:
        plt.ioff()
print("Test 10 done — if sliders vanished, matplotlib sync redraw is the culprit.")


=== Test 10: mix + immediate plt.close('all') after display ===


Test 10 done — if sliders vanished, matplotlib sync redraw is the culprit.


In [20]:
# Test 9 — real tier1_cell10_playground_run.py (same exec as 538D CELL 10)
import sys
from pathlib import Path

print("=== Test 9: real tier1_cell10_playground_run.py ===")
globals().pop("_CELL10_PLAYGROUND_LIVE", None)

_p = Path("tier1_cell10_playground_run.py")
if not _p.is_file():
    _p = Path("sports/tier1_cell10_playground_run.py")
if not _p.is_file() and Path.cwd().name == "sports":
    _p = Path("tier1_cell10_playground_run.py")
if not _p.is_file():
    raise FileNotFoundError(f"Cannot find tier1_cell10_playground_run.py (cwd={Path.cwd()})")

exec(compile(_p.read_text(encoding="utf-8"), str(_p.resolve()), "exec"), globals())
print("Test 9 done — compare slider visibility to Tests 2–5 and 8.")


=== Test 9: real tier1_cell10_playground_run.py ===


CELL 10: panel displayed — plots load shortly (or click Run / refresh plot)
Test 9 done — compare slider visibility to Tests 2–5 and 8.


In [21]:
# Test 11 — generative_eda import before widgets
from pathlib import Path

print("=== Test 11: generative_eda import before widgets ===")
_p = Path("sports/tier1_cell10_widget_probe_generative_import.py")
if not _p.is_file():
    _p = Path("tier1_cell10_widget_probe_generative_import.py")
exec(compile(_p.read_text(encoding="utf-8"), str(_p), "exec"), globals())

=== Test 11: generative_eda import before widgets ===
=== Test 11: generative_eda import before widgets ===


Test 11 done — sliders visible?


In [ ]:
# BISECT — incremental playground rebuild (re-run THIS cell only after each .py edit)
from pathlib import Path

print("=== BISECT: tier1_cell10_playground_bisect.py ===")
_p = Path("sports/tier1_cell10_playground_bisect.py")
if not _p.is_file():
    _p = Path("tier1_cell10_playground_bisect.py")
if not _p.is_file():
    raise FileNotFoundError(f"Missing tier1_cell10_playground_bisect.py (cwd={Path.cwd()})")
exec(compile(_p.read_text(encoding="utf-8"), str(_p.resolve()), "exec"), globals())
